# Валидация метрик аномального режима (гир 1.5)

Свечи `5m` (OHLC+volume) из offline-дампов OKX / Bybit + подсветка окон `regime_on` по канону [`docs/regime-metrics-v0.md`](../docs/regime-metrics-v0.md).

**График:** TradingView Lightweight Charts через `lightweight-charts` (`JupyterChart`) — scroll / pan / zoom как в TradingView. Аномальные окна — оранжевые (OKX) / синие (Bybit) вертикальные полосы на барах с `regime_on`.

Данные (окно по умолчанию ~1 месяц):
- `output/okx_bar5m_hist_regime/` — `[2026-07-08, 2026-08-08)`
- `output/bybit_bar5m_hist_regime/` — то же

Формулы: `research/regime_metrics.py`. Пороги экспертные — крутите в CONFIG.

Запуск: ядро из `venv` репозитория (`/Users/mishatrubik/Desktop/spread/venv`). Пакет: `lightweight-charts` (2.x).

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from lightweight_charts import JupyterChart

REPO = Path("..").resolve()
if not (REPO / "research" / "regime_metrics.py").exists():
    REPO = Path.cwd().resolve()
sys.path.insert(0, str(REPO))

from research.regime_metrics import (
    RegimeParams,
    build_regime_frame,
    regime_episodes,
    sanity_summary,
    _rolling_z,
)

OKX_ROOT = REPO / "output" / "okx_bar5m_hist_regime"
BYBIT_ROOT = REPO / "output" / "bybit_bar5m_hist_regime"

print("REPO", REPO)
print("okx exists", OKX_ROOT.exists(), "bybit exists", BYBIT_ROOT.exists())
print("lightweight-charts JupyterChart OK")

## CONFIG

Одна монета на скроллируемый график + окно месяца + пороги режима.

In [ ]:
# --- выбор данных ---
COIN = "BTC"  # primary coin for the scrollable chart
START = "2026-07-08"  # UTC, inclusive (event_date)
END = "2026-08-08"  # UTC, exclusive

# "okx" | "bybit" | "both" (свечи primary; зоны regime_on обеих бирж)
EXCHANGE_MODE = "both"
PRIMARY_EXCHANGE = "okx"  # свечи / volume / z pane

# --- пороги режима (ручные, volume z_smooth hysteresis) ---
PARAMS = RegimeParams(
    W=48,
    W_min=48,
    W_s=3,
    Z_enter=2.0,
    Z_exit=1.0,
    K_persist=6,
    P_persist=0.5,
    require_persistence=False,
    Z_amp=2.0,
    require_amp=False,  # True → AND с z по amp_ohlc
)

# опциональный amp gate поверх volume regime (если require_amp=False)
USE_OHLC_AMP_GATE = False

CHART_WIDTH = 1100
CHART_HEIGHT = 640

print("coin", COIN)
print("window", START, "→", END)
print("mode", EXCHANGE_MODE, "primary", PRIMARY_EXCHANGE)
print("params", asdict(PARAMS))
print("amp_gate", USE_OHLC_AMP_GATE or PARAMS.require_amp)

In [ ]:
def load_hist_bars(root: Path, base_coin: str, start: str, end: str) -> pd.DataFrame:
    coin_dir = root / f"base_coin={base_coin}"
    if not coin_dir.exists():
        raise FileNotFoundError(coin_dir)
    days = sorted(
        p.name.split("=", 1)[1]
        for p in coin_dir.glob("event_date=*")
        if p.is_dir()
    )
    days = [d for d in days if start <= d < end]
    if not days:
        raise FileNotFoundError(f"no days for {base_coin} in [{start},{end}) under {root}")
    parts = []
    for d in days:
        path = coin_dir / f"event_date={d}" / "part.parquet"
        if path.exists():
            parts.append(pd.read_parquet(path))
    df = pd.concat(parts, ignore_index=True)
    df = df.sort_values("bar_start_ts_ms", kind="mergesort").drop_duplicates(
        subset=["bar_start_ts_ms"], keep="last"
    )
    return df.reset_index(drop=True)


def attach_regime(
    bars: pd.DataFrame, params: RegimeParams, *, use_ohlc_amp_gate: bool
) -> pd.DataFrame:
    """Volume regime (+ optional AND with z of amp_ohlc)."""
    frame = build_regime_frame(bars, params=params, ticks=None)
    if "amp_ohlc" in bars.columns:
        frame["amp_ohlc"] = bars["amp_ohlc"].to_numpy()
        frame["z_amp_ohlc"] = _rolling_z(
            frame["amp_ohlc"], params.W, params.W_min, params.eps
        ).to_numpy()
    else:
        frame["z_amp_ohlc"] = np.nan

    if use_ohlc_amp_gate or params.require_amp:
        amp_ok = frame["z_amp_ohlc"] >= params.Z_amp
        frame["regime_on"] = frame["regime_on"] & amp_ok.fillna(False)

    frame["bar_dt"] = pd.to_datetime(frame["bar_start_ts_ms"], unit="ms", utc=True)
    return frame


def to_tv_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:
    """Columns expected by lightweight-charts: time + OHLCV."""
    out = pd.DataFrame(
        {
            "time": frame["bar_dt"].dt.tz_convert("UTC").dt.tz_localize(None),
            "open": frame["open"].astype(float),
            "high": frame["high"].astype(float),
            "low": frame["low"].astype(float),
            "close": frame["close"].astype(float),
            "volume": frame["volume"].astype(float),
        }
    )
    return out


def episode_spans(frame: pd.DataFrame) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    eps = regime_episodes(frame["regime_on"], frame["bar_start_ts_ms"])
    spans = []
    for _, row in eps.iterrows():
        t0 = pd.to_datetime(int(row["start_ts_ms"]), unit="ms", utc=True).tz_localize(None)
        t1 = pd.to_datetime(int(row["end_ts_ms"]), unit="ms", utc=True).tz_localize(None)
        spans.append((t0, t1))
    return spans


def load_frames(
    coin: str,
    *,
    exchange_mode: str,
    params: RegimeParams,
    use_ohlc_amp_gate: bool,
) -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    if exchange_mode in ("okx", "both"):
        raw = load_hist_bars(OKX_ROOT, coin, START, END)
        frames["okx"] = attach_regime(raw, params, use_ohlc_amp_gate=use_ohlc_amp_gate)
    if exchange_mode in ("bybit", "both"):
        raw = load_hist_bars(BYBIT_ROOT, coin, START, END)
        frames["bybit"] = attach_regime(raw, params, use_ohlc_amp_gate=use_ohlc_amp_gate)
    return frames

## Scrollable chart (TradingView Lightweight Charts)

- Свечи + volume: primary биржа
- Нижняя панель: `z_vol_smooth` + пороги enter/exit
- **Аномальные зоны:** полупрозрачные вертикальные полосы на барах, где `regime_on` (гистерезис по `z_vol_smooth`). Оранжевый = OKX, синий = Bybit. В легенде серии `regime:okx` / `regime:bybit`.

> Технически: `vertical_span` в `lightweight-charts` 2.x сломан (JS `calculateTrendLine` отсутствует). Поэтому зоны рисуются histogram-overlay на отдельной price scale (полная высота панели) — визуально те же shaded time ranges.

Pan влево/вправо и zoom колёсиком / pinch — как в TradingView.

In [ ]:
REGIME_COLORS = {
    "okx": "rgba(255, 165, 0, 0.32)",
    "bybit": "rgba(70, 130, 180, 0.32)",
}


def overlay_regime_on(
    chart: JupyterChart,
    frames: dict[str, pd.DataFrame],
) -> dict[str, dict[str, int]]:
    """Shade bars where regime_on is True (full-height histogram on own price scale).

    Do not use chart.vertical_span — broken in lightweight-charts 2.x
    (calls missing JS calculateTrendLine).
    """
    stats: dict[str, dict[str, int]] = {}
    for ex, fr in frames.items():
        name = f"regime:{ex}"
        color = REGIME_COLORS.get(ex, "rgba(252, 219, 3, 0.28)")
        hist = chart.create_histogram(
            name=name,
            color=color,
            price_line=False,
            price_label=False,
            scale_margin_top=0.0,
            scale_margin_bottom=0.0,
        )
        on = fr["regime_on"].fillna(False)
        n_eps = int(len(regime_episodes(fr["regime_on"], fr["bar_start_ts_ms"])))
        if not on.any():
            hist.set(pd.DataFrame({"time": pd.Series(dtype="datetime64[ns]"), name: pd.Series(dtype=float)}))
            stats[ex] = {"on_bars": 0, "episodes": 0}
            continue
        hist.set(
            pd.DataFrame(
                {
                    "time": fr.loc[on, "bar_dt"].dt.tz_convert("UTC").dt.tz_localize(None),
                    name: 1.0,
                }
            )
        )
        stats[ex] = {"on_bars": int(on.sum()), "episodes": n_eps}
    return stats


def build_regime_chart(
    coin: str,
    frames: dict[str, pd.DataFrame],
    *,
    primary: str,
    params: RegimeParams,
    width: int = 1100,
    height: int = 640,
) -> tuple[JupyterChart, dict[str, dict[str, int]]]:
    if primary not in frames:
        primary = next(iter(frames))
    main = frames[primary]

    chart = JupyterChart(width=width, height=height, toolbox=False)
    chart.layout(background_color="#111111", text_color="#DDDDDD")
    chart.candle_style(
        up_color="#26a69a",
        down_color="#ef5350",
        border_up_color="#26a69a",
        border_down_color="#ef5350",
        wick_up_color="#26a69a",
        wick_down_color="#ef5350",
    )
    chart.volume_config(
        scale_margin_top=0.8,
        scale_margin_bottom=0.0,
        up_color="rgba(38,166,154,0.45)",
        down_color="rgba(239,83,80,0.45)",
    )
    chart.legend(visible=True, ohlc=True, percent=True, lines=True)
    chart.watermark(
        f"{coin} · {primary.upper()} 5m · orange=OKX regime · blue=Bybit regime",
        font_size=22,
        color="rgba(180,180,200,0.28)",
    )
    chart.time_scale(right_offset=4, time_visible=True, seconds_visible=False)

    chart.set(to_tv_ohlcv(main))
    overlay_stats = overlay_regime_on(chart, frames)

    # bottom pane: z_vol_smooth for primary (+ optional secondary as line overlay on same subchart)
    z_pane = chart.create_subchart(position="bottom", width=1.0, height=0.28, sync=True)
    z_pane.layout(background_color="#111111", text_color="#DDDDDD")
    # same anomaly shading on z pane (synced time axis)
    overlay_regime_on(z_pane, frames)
    z_line = z_pane.create_line(name=f"z_smooth:{primary}", color="#ffcc66", width=2)
    z_df = pd.DataFrame(
        {
            "time": main["bar_dt"].dt.tz_convert("UTC").dt.tz_localize(None),
            f"z_smooth:{primary}": main["z_vol_smooth"].astype(float),
        }
    ).dropna()
    z_line.set(z_df)
    z_pane.horizontal_line(params.Z_enter, color="#ef5350", width=1, style="dotted", text="Z_enter")
    z_pane.horizontal_line(params.Z_exit, color="#9e9e9e", width=1, style="dashed", text="Z_exit")

    if len(frames) > 1:
        for ex, fr in frames.items():
            if ex == primary:
                continue
            line = z_pane.create_line(name=f"z_smooth:{ex}", color="#64b5f6", width=1)
            df = pd.DataFrame(
                {
                    "time": fr["bar_dt"].dt.tz_convert("UTC").dt.tz_localize(None),
                    f"z_smooth:{ex}": fr["z_vol_smooth"].astype(float),
                }
            ).dropna()
            line.set(df)

    return chart, overlay_stats


frames = load_frames(
    COIN,
    exchange_mode=EXCHANGE_MODE,
    params=PARAMS,
    use_ohlc_amp_gate=USE_OHLC_AMP_GATE,
)

for ex, fr in frames.items():
    days = (
        pd.to_datetime(fr["bar_start_ts_ms"], unit="ms", utc=True)
        .dt.strftime("%Y-%m-%d")
        .nunique()
    )
    print(
        f"{COIN} {ex}: bars={len(fr)} days={days} "
        f"[{fr['bar_dt'].iloc[0]} → {fr['bar_dt'].iloc[-1]}]"
    )
    print(" ", sanity_summary(fr))

primary = PRIMARY_EXCHANGE if PRIMARY_EXCHANGE in frames else next(iter(frames))
chart, overlay_stats = build_regime_chart(
    COIN,
    frames,
    primary=primary,
    params=PARAMS,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
)
print(
    "shaded zones: orange=OKX regime_on · blue=Bybit regime_on · "
    f"candles/volume={primary}"
)
print("overlay_stats", overlay_stats)
chart.load()

## Таблица эпизодов

Сводка по выделенным аномальным окнам (удобно сверять с глазом на графике).

In [ ]:
rows = []
for ex, fr in frames.items():
    eps = regime_episodes(fr["regime_on"], fr["bar_start_ts_ms"])
    if eps.empty:
        continue
    eps = eps.copy()
    eps["base_coin"] = COIN
    eps["exchange"] = ex
    eps["start_dt"] = pd.to_datetime(eps["start_ts_ms"], unit="ms", utc=True)
    eps["end_dt"] = pd.to_datetime(eps["end_ts_ms"], unit="ms", utc=True)
    eps["duration_min"] = (eps["end_ts_ms"] - eps["start_ts_ms"]) / 60_000
    rows.append(eps)

episodes_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if episodes_df.empty:
    print("no regime episodes in window")
else:
    display(
        episodes_df.sort_values(["start_ts_ms", "exchange"]).reset_index(drop=True)
    )

## Быстрая сверка OKX vs Bybit

Доля баров, где режим включён только на одной бирже — кандидаты на «односторонний крупный поток», а не общий стресс монеты.

In [ ]:
if EXCHANGE_MODE == "both" and "okx" in frames and "bybit" in frames:
    a = frames["okx"][["bar_start_ts_ms", "regime_on"]].rename(columns={"regime_on": "on_okx"})
    b = frames["bybit"][["bar_start_ts_ms", "regime_on"]].rename(columns={"regime_on": "on_bybit"})
    m = a.merge(b, on="bar_start_ts_ms", how="inner")
    n = len(m)
    if n == 0:
        print("no overlapping bars")
    else:
        display(
            pd.DataFrame(
                [
                    {
                        "base_coin": COIN,
                        "n_bars": n,
                        "both_on": float((m.on_okx & m.on_bybit).mean()),
                        "okx_only": float((m.on_okx & ~m.on_bybit).mean()),
                        "bybit_only": float((m.on_bybit & ~m.on_okx).mean()),
                        "either": float((m.on_okx | m.on_bybit).mean()),
                    }
                ]
            )
        )
else:
    print("Включите EXCHANGE_MODE='both' для сравнения бирж.")